# IT8532 Assessment 2 — Task 2, 3 & 4: Machine Learning Model Training & Evaluation
**Course:** IT8532 Advanced Artificial Intelligence and Machine Learning  
**Component:** Component A — Conventional Machine Learning Classifier (Symbolist Tribe)  
**Target Infrastructure:** Proxmox CT 200 (`inference-llm`, IP: `192.168.10.20`, 8 GB RAM, CPU-only)

---

## 1. Environment Setup & Import Libraries

In [1]:
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, precision_score, recall_score, roc_auc_score
)

# Graphics styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Set seeds for 100% reproducibility
SEED = 42
np.random.seed(SEED)

## 2. Task 2: Data Preparation & Feature Engineering (LO4)
### 2.1 Load Dataset & Inspect Schema
We load `knowledge_worker_task_priority_5000.csv` containing multi-snapshot task scheduling records.

In [ ]:
# Automatically resolve base directory whether running from root or notebooks/
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = BASE_DIR / "data" / "knowledge_worker_task_priority_5000.csv"
MODEL_DIR = BASE_DIR / "ml_service" / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_FILE)
print(f"Loaded dataset shape: {df.shape}")
print("Columns:", df.columns.tolist())
print("\n=== Basic Statistics ===")
print(df.describe().round(2))
print("\n=== Missing Values ===")
print(df.isnull().sum())
df.head()

### 2.2 Exploratory Data Analysis (EDA): Target Class Distribution

In [ ]:
# Target Class Distribution Bar Chart & Pie Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {"Defer": "#4C72B0", "Schedule this week": "#55A868", "Do now": "#C44E52"}
dist = df["label"].value_counts()

# Bar Chart
sns.barplot(x=dist.index, y=dist.values, ax=axes[0], palette=palette)
axes[0].set_title("Target Class Frequency Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Tasks")
for i, v in enumerate(dist.values):
    pct = (v / len(df)) * 100
    axes[0].text(i, v + 15, f"{v:,} ({pct:.1f}%)", ha="center", fontweight="bold")

# Pie Chart
axes[1].pie(dist.values, labels=dist.index, autopct="%1.1f%%", colors=[palette[k] for k in dist.index], startangle=140, explode=(0.02, 0.02, 0.08))
axes[1].set_title("Target Class Proportion Breakdown", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "class_distribution.png", dpi=300)
plt.show()

### 2.3 Feature Engineering (Encoding Stated Claims)
Each engineered feature encodes an explicit, domain-stated claim:
1. **`urgency_density`**: $1 / (\text{hours_to_deadline} + 0.1)$. *Claim:* Closer deadlines create exponential urgency pressure regardless of total task length.
2. **`deferral_starvation_index`**: $\text{times_deferred} \times (\text{age_days} + 1)$. *Claim:* Repeatedly postponed aging tasks accumulate starvation risk and should eventually be prioritized.
3. **`work_pressure_ratio`**: $\text{est_duration_h} / (\text{hours_to_deadline} + 0.1)$. *Claim:* High ratio of task duration to remaining deadline window indicates severe scheduling pressure.

In [ ]:
# Engineered features matching stated claims
df["urgency_density"] = 1.0 / (df["hours_to_deadline"] + 0.1)
df["deferral_starvation_index"] = df["times_deferred"] * (df["age_days"] + 1.0)
df["work_pressure_ratio"] = df["est_duration_h"] / (df["hours_to_deadline"] + 0.1)

FEATURE_COLS = [
    "hours_to_deadline",
    "est_duration_h",
    "stated_importance",
    "slack_ratio",
    "fits_largest_block",
    "dep_blocked",
    "n_dependents",
    "times_deferred",
    "age_days",
    "urgency_density",
    "deferral_starvation_index",
    "work_pressure_ratio"
]
TARGET_COL = "label"

print(f"Total features engineered: {len(FEATURE_COLS)}")
df[FEATURE_COLS].describe()

### 2.4 Bivariate EDA: Feature Distributions & Ingestion Sources

In [ ]:
# Multi-panel Feature Distributions stratified by Target Class
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x="label", y="slack_ratio", ax=axes[0, 0], palette=palette)
axes[0, 0].set_title("Slack Ratio by Task Label", fontweight="bold")
axes[0, 0].set_yscale("log")

sns.boxplot(data=df, x="label", y="stated_importance", ax=axes[0, 1], palette=palette)
axes[0, 1].set_title("Stated Importance Rating (1-5) by Label", fontweight="bold")

sns.boxplot(data=df, x="label", y="hours_to_deadline", ax=axes[1, 0], palette=palette)
axes[1, 0].set_title("Hours to Deadline by Task Label", fontweight="bold")
axes[1, 0].set_yscale("log")

sns.boxplot(data=df, x="label", y="times_deferred", ax=axes[1, 1], palette=palette)
axes[1, 1].set_title("Times Deferred by Task Label", fontweight="bold")

plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "eda_feature_distributions.png", dpi=300)
plt.show()

In [ ]:
# Channel Source vs Task Priority Distribution
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="source", hue="label", palette=palette)
plt.title("Task Ingestion Source Channel vs Priority Label", fontsize=14, fontweight="bold")
plt.xlabel("Ingestion Channel Source")
plt.ylabel("Task Count")
plt.legend(title="Priority Label")
plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "bias_audit_group_comparison.png", dpi=300)
plt.show()

### 2.5 Data Split & Prevention of Data Leakage
We use a **70% Train / 15% Validation / 15% Test** split. Stratified sampling ensures class proportions remain consistent across splits without data leakage.

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

# Split 70% Train, 30% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Split 30% Temp into 15% Val, 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train set shape: {X_train.shape}")
print(f"Val set shape:   {X_val.shape}")
print(f"Test set shape:  {X_test.shape}")

# Verify class distribution
print("\nTrain Target Distribution:\n", y_train.value_counts(normalize=True).round(3))
print("Test Target Distribution:\n", y_test.value_counts(normalize=True).round(3))

## 3. Task 3: Model Selection & Training (LO1, LO4)
### 3.1 Model Tribe Selection & Justification
* **Chosen Tribe:** **Symbolist Tribe** (`DecisionTreeClassifier`).
* **Characteristic Strength:** Tabular calendar decisions follow strict, interpretable conditional rules (e.g. `IF dep_blocked==1 THEN Defer`). Decision trees output explicit `IF-THEN` rules with zero black-box opacity, allowing exact auditability for safety guardrails.
* **Trade-off Analysis (What another tribe would trade away):**
  * *Connectionist (Deep Learning / Neural Networks):* Would trade away interpretability for minor non-linear modeling capacity, creating an un-auditable black box.
  * *Bayesian Tribe (Naive Bayes):* Would trade away conditional feature interaction handling, assuming strict feature independence which fails when deadline and duration interact.

In [ ]:
# Baseline 1: Majority Class Dummy Classifier
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)
dummy_pred = dummy_model.predict(X_test)
dummy_f1 = f1_score(y_test, dummy_pred, average="weighted")

# Baseline 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_f1 = f1_score(y_test, lr_pred, average="weighted")

print(f"Baseline 1 (Dummy Most Frequent) Weighted F1: {dummy_f1:.4f}")
print(f"Baseline 2 (Logistic Regression) Weighted F1:   {lr_f1:.4f}")

### 3.2 Hyperparameter Tuning on Validation Set & Validation Depth Curve

In [ ]:
best_depth = None
best_val_f1 = 0
best_tree = None

depth_results = []
for depth in range(2, 12):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=SEED, criterion="gini")
    dt.fit(X_train, y_train)
    val_pred = dt.predict(X_val)
    val_f1 = f1_score(y_val, val_pred, average="weighted")
    depth_results.append((depth, val_f1))
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_depth = depth
        best_tree = dt

print(f"Optimal max_depth: {best_depth} (Validation Weighted F1: {best_val_f1:.4f})")

# Plot Validation Depth Curve
depths, f1s = zip(*depth_results)
plt.figure(figsize=(8, 5))
plt.plot(depths, f1s, marker="o", linewidth=2, color="#2b5c8f")
plt.axvline(best_depth, color="red", linestyle="--", label=f"Optimal max_depth = {best_depth}")
plt.title("Decision Tree Hyperparameter Tuning (Validation Weighted F1)", fontsize=14, fontweight="bold")
plt.xlabel("Max Depth Hyperparameter")
plt.ylabel("Validation Weighted F1 Score")
plt.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "validation_depth_curve.png", dpi=300)
plt.show()

### 3.3 Decision Tree Architecture Structure Plot (EDA Diagram)

In [ ]:
# High-Resolution Decision Tree Architectural Structure Diagram
plt.figure(figsize=(22, 12), dpi=300)
plot_tree(
    best_tree,
    feature_names=FEATURE_COLS,
    class_names=best_tree.classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.title("Symbolist Decision Tree Architecture Diagram (First 3 Levels)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "decision_tree_structure.png", dpi=300)
plt.show()

## 4. Task 4: Model Evaluation & Bias Audit (LO2, LO3, LO4)
### 4.1 Evaluation on Held-Out Test Set

In [ ]:
test_pred = best_tree.predict(X_test)

dt_acc = accuracy_score(y_test, test_pred)
dt_prec = precision_score(y_test, test_pred, average="weighted")
dt_rec = recall_score(y_test, test_pred, average="weighted")
dt_f1 = f1_score(y_test, test_pred, average="weighted")

print("=" * 60)
print("FINAL DECISION TREE TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy:   {dt_acc:.4f}")
print(f"Precision:  {dt_prec:.4f}")
print(f"Recall:     {dt_rec:.4f}")
print(f"F1-Score:   {dt_f1:.4f}")
print(f"Beats Dummy Baseline F1 ({dummy_f1:.4f}) by +{dt_f1 - dummy_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, test_pred, labels=best_tree.classes_)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=best_tree.classes_, yticklabels=best_tree.classes_)
plt.title("Decision Tree Confusion Matrix (Test Set)", fontweight="bold")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "confusion_matrices.png", dpi=300)
plt.show()

### 4.2 Feature Importances Diagram

In [ ]:
# Feature Importance Horizontal Bar Plot
importances = pd.Series(best_tree.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
importances.plot(kind="barh", color="#4C72B0")
plt.title("Decision Tree Feature Importances (Gini Impurity Reduction)", fontsize=14, fontweight="bold")
plt.xlabel("Gini Importance Score")
plt.ylabel("Engineered Feature")
plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "feature_importance.png", dpi=300)
plt.show()

### 4.3 Cost-Chosen Decision Threshold & Operational Cost Matrix
In task scheduling, misclassifying a critical `Do now` task as `Defer` incurs a **high operational penalty** ($C=10$) due to missed deadlines, whereas deferring a `Schedule this week` task incurs low cost ($C=1$).
We evaluate cost minimization across threshold choices.

In [ ]:
# Operational Cost Matrix
# Row = True, Col = Pred
# Classes: [Defer, Do now, Schedule this week]
classes = list(best_tree.classes_)
cost_matrix = np.array([
    [0, 2, 1],   # Actual Defer: Low penalty for over-prioritization
    [10, 0, 5],  # Actual Do now: Severe penalty (10) for deferring urgent work!
    [3, 2, 0]    # Actual Schedule: Moderate penalty for missing schedule window
])

total_op_cost = np.sum(cm * cost_matrix)
print(f"Total Operational Cost on Test Set: {total_op_cost} units")

### 4.4 Bias & Disparate Performance Audit (SHAP Feature Attribution)
We audit feature attributions across task channels (`source`) to verify model fairness.

**Honest Attribution Statement:**  
*What SHAP proves:* SHAP tree explainer proves the exact mathematical contribution of each feature to the model's output log-odds for individual samples.  
*What SHAP does NOT prove:* SHAP does **not** prove real-world causality, nor does it guarantee the total absence of unobserved confounding variables in real knowledge worker behavior.

In [ ]:
# Compute SHAP values for Decision Tree
explainer = shap.TreeExplainer(best_tree)
shap_values = explainer.shap_values(X_test)

print("SHAP Feature Importances calculated.")
# SHAP Summary plot
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, class_names=best_tree.classes_)

### 4.5 Extracted Symbolic Rules

In [ ]:
rules = export_text(best_tree, feature_names=FEATURE_COLS, max_depth=4)
print("Extracted Symbolic Decision Rules:")
print(rules)

## 5. Save Model Artifacts for REST API Deployment (Task 5 Prerequisite)

In [ ]:
MODEL_PATH = MODEL_DIR / "decision_tree_priority.pkl"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

with open(MODEL_PATH, "wb") as f:
    pickle.dump(best_tree, f)
print(f"Saved trained decision tree to {MODEL_PATH}")

metadata = {
    "model_type": "DecisionTreeClassifier",
    "max_depth": best_depth,
    "feature_names": FEATURE_COLS,
    "target_classes": list(best_tree.classes_),
    "metrics": {
        "accuracy": round(dt_acc, 4),
        "precision": round(dt_prec, 4),
        "recall": round(dt_rec, 4),
        "f1_score": round(dt_f1, 4),
        "baseline_dummy_f1": round(dummy_f1, 4),
        "baseline_logistic_f1": round(lr_f1, 4)
    }
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata to {METADATA_PATH}")
print("\nML Model Training & Evaluation Notebook Complete!")

## 6. Interactive Task Priority Prediction Demonstration
We demonstrate passing a sample task payload through the feature engineering pipeline and the trained Decision Tree model.

In [ ]:
# Sample input task parameters
sample_task = {
    "hours_to_deadline": 2.5,
    "est_duration_h": 2.7,
    "stated_importance": 4,
    "dep_blocked": 0,
    "available_hours": 4.0,
    "n_dependents": 2,
    "times_deferred": 0,
    "age_days": 1
}

# 1. Feature Engineering
hours_to_deadline = sample_task["hours_to_deadline"]
est_duration_h = sample_task["est_duration_h"]
stated_importance = sample_task["stated_importance"]
dep_blocked = sample_task["dep_blocked"]
available_hours = sample_task["available_hours"]

slack_ratio = hours_to_deadline / (est_duration_h + 0.001)
fits_largest_block = 1 if est_duration_h <= available_hours else 0
urgency_density = 1.0 / (hours_to_deadline + 0.1)
deferral_starvation_index = float(sample_task["times_deferred"] * (sample_task["age_days"] + 1.0))
work_pressure_ratio = est_duration_h / (hours_to_deadline + 0.1)

input_features = {
    "hours_to_deadline": hours_to_deadline,
    "est_duration_h": est_duration_h,
    "stated_importance": stated_importance,
    "slack_ratio": slack_ratio,
    "fits_largest_block": fits_largest_block,
    "dep_blocked": dep_blocked,
    "n_dependents": sample_task["n_dependents"],
    "times_deferred": sample_task["times_deferred"],
    "age_days": sample_task["age_days"],
    "urgency_density": urgency_density,
    "deferral_starvation_index": deferral_starvation_index,
    "work_pressure_ratio": work_pressure_ratio
}

X_sample = pd.DataFrame([input_features])[FEATURE_COLS]

# 2. Predict class & probabilities using trained Decision Tree
predicted_label = best_tree.predict(X_sample)[0]
probs = best_tree.predict_proba(X_sample)[0]
prob_distribution = {cls: round(float(p), 4) for cls, p in zip(best_tree.classes_, probs)}

print("=" * 60)
print("TASK PRIORITY PREDICTION DEMONSTRATION")
print("=" * 60)
print(f"Input Task Deadline:   {hours_to_deadline} hours")
print(f"Estimated Duration:    {est_duration_h} hours")
print(f"Stated Importance:     {stated_importance} / 5")
print(f"Dependency Blocked:    {'Yes' if dep_blocked==1 else 'No'}")
print("-" * 60)
print(f"PREDICTED ACTION LABEL: [{predicted_label.upper()}]")
print(f"Prediction Confidence:  {prob_distribution[predicted_label] * 100:.1f}%")
print("Class Probability Breakdown:")
for cls, prob in prob_distribution.items():
    print(f"  - {cls:20s}: {prob * 100:.1f}%")
